# VRP Toolkit Quickstart Tutorial

This tutorial demonstrates the basic workflow of the VRP Toolkit for solving Pickup and Delivery Problems with Time Windows (PDPTW) using the Adaptive Large Neighborhood Search (ALNS) algorithm.

## Overview

We'll walk through:
1. **Setup**: Import necessary modules and set up the environment
2. **Synthetic Map**: Create a synthetic map with restaurants and customers
3. **Demand Generation**: Generate delivery demands across time intervals
4. **PDPTW Instance**: Create a complete PDPTW problem instance
5. **ALNS Solution**: Solve the instance using ALNS algorithm
6. **Visualization**: Visualize the solution

Let's get started!

## 1. Setup and Imports

First, import the necessary modules from the VRP Toolkit.

In [ ]:
# Import core modules from VRP Toolkit
from vrp_toolkit.data.map import RealMap
from vrp_toolkit.data.generators import DemandGenerator, OrderGenerator
from vrp_toolkit.problems.pdptw import PDPTWInstance
from vrp_toolkit.algorithms.alns.solver import ALNS, ALNSConfig, greedy_insertion_initial_solution

# Import standard libraries
import numpy as np
import pandas as pd
import random

# Set random seed for reproducibility
seed_value = 42
np.random.seed(seed_value)
random.seed(seed_value)

print("Imports successful! VRP Toolkit is ready.")

## 2. Create Synthetic Map

We'll create a synthetic map with 2 restaurants and 4 customers. The `RealMap` class generates random coordinates and computes Euclidean distances between nodes.

In [ ]:
# Create a synthetic map with 2 restaurants and 4 customers
real_map = RealMap(
    n_r=2,
    n_c=4,
    dist_function=np.random.uniform,
    dist_params={'low': -1, 'high': 1}
)

# Display basic map information
print(f"Map created successfully!")
print(f"• Number of restaurants: {real_map.N_R}")
print(f"• Number of customers: {real_map.N_C}")
print(f"• Total nodes (including depot/destination/charging): {len(real_map.all_nodes)}")
print(f"• Restaurant indices: {real_map.restaurants}")
print(f"• Customer indices: {real_map.customers}")

## 3. Generate Delivery Demands

Next, we generate delivery demands between restaurant-customer pairs across time intervals. The `DemandGenerator` creates synthetic demand data suitable for PDPTW problems.

In [ ]:
# Configure demand generation parameters
random_params = {
    'sample_dist': {'function': np.random.randint, 'params': {'low': 1, 'high': 3}},
    'demand_dist': {'function': np.random.poisson, 'params': {'lam': 2}}
}

# Create demand generator
demands = DemandGenerator(
    time_range=30,          # Total time range in minutes
    time_step=10,           # Time interval step size
    restaurants=real_map.restaurants,
    customers=real_map.customers,
    random_params=random_params
)

# Display demand information
print(f"Demand generation complete!")
print(f"• Time intervals: {demands.time_intervals}")
print(f"• Restaurant-customer pairs: {len(demands.pairs)}")
print(f"\nSample of demand table:")
print(demands.demand_table.head())

## 4. Create PDPTW Instance

Now we create a complete PDPTW instance using the map and demand data. The `OrderGenerator` converts demand information into a complete order table with pickup-delivery pairs, time windows, and service times.

In [ ]:
# Time window parameters
time_params = {
    'time_window_length': 30,  # Length of delivery time windows (minutes)
    'service_time': 5,         # Service time required at each node (minutes)
    'extra_time': 10           # Extra buffer time added to delivery start
}

# Create order generator
pdptw_order = OrderGenerator(
    real_map=real_map,
    demand_table=demands.demand_table,
    time_params=time_params,
    robot_speed=4              # Robot speed in distance units per minute
)

# Get the generated order table
order_table = pdptw_order.get_order_table()

# Create PDPTW instance
pdptw_instance = PDPTWInstance(
    order_table=order_table,
    distance_matrix=real_map.distance_matrix,
    time_matrix=real_map.distance_matrix / 4.0,  # Convert distance to time using robot speed
    robot_speed=4.0
)

# Display instance information
print(f"PDPTW instance created successfully!")
print(f"• Total orders: {pdptw_instance.n}")
print(f"• Total nodes: {len(pdptw_instance.indices)}")
print(f"• Distance matrix shape: {pdptw_instance.distance_matrix.shape}")
print(f"\nSample of order table:")
print(order_table.head())

## 5. Solve with ALNS Algorithm

Now we solve the PDPTW instance using the Adaptive Large Neighborhood Search (ALNS) algorithm. We'll:
1. Configure ALNS parameters
2. Generate an initial solution using greedy insertion
3. Run ALNS optimization

In [ ]:
# Problem parameters
num_vehicles = 4
vehicle_capacity = 6
battery_consume_rate = 1
penalty_unvisited = 100
penalty_delayed = 15

# Battery capacity calculation
battery = 8
if_battery_relaxation = 1

# Simple battery capacity calculation
def battery_relaxation(battery, dist_matrix, robot_speed, indicator=None):
    if indicator:
        battery_capacity = (battery - np.mean(dist_matrix[0][1:-1])) * 2 / robot_speed * 60
    else:
        battery_capacity = battery / robot_speed * 60
    return battery_capacity

battery_capacity = battery_relaxation(
    battery,
    pdptw_instance.distance_matrix,
    pdptw_instance.robot_speed,
    if_battery_relaxation
)

# Generate initial solution using greedy insertion
initial_solution = greedy_insertion_initial_solution(
    instance=pdptw_instance,
    num_vehicles=num_vehicles,
    vehicle_capacity=vehicle_capacity,
    battery_capacity=battery_capacity,
    battery_consume_rate=battery_consume_rate,
    penalty_unvisited=penalty_unvisited,
    penalty_delayed=penalty_delayed
)

print(f"Initial solution generated!")
print(f"• Objective value: {initial_solution.objective_value():.2f}")
print(f"• Is feasible: {initial_solution.is_feasible()}")

In [ ]:
# Configure ALNS algorithm
alns_config = ALNSConfig(
    max_iterations=100,
    segment_length=10,
    num_segments=5,
    start_temp=10000,
    cooling_rate=0.99,
    num_removal=5
)

# Create ALNS solver
alns_solver = ALNS(
    initial_solution=initial_solution,
    config=alns_config,
    dist_matrix=pdptw_instance.distance_matrix,
    battery_capacity=battery_capacity
)

# Run ALNS optimization
print("Starting ALNS optimization...")
best_solution, best_objective, history = alns_solver.solve()

print(f"\nOptimization complete!")
print(f"• Best objective value: {best_objective:.2f}")
print(f"• Improvement: {initial_solution.objective_value() - best_objective:.2f}")
print(f"• Solution feasible: {best_solution.is_feasible()}")

## 6. Visualization

Finally, we can visualize the solution. Note: This requires matplotlib to be installed.

In [ ]:
try:
    import matplotlib.pyplot as plt
    
    # Plot optimization history
    plt.figure(figsize=(10, 6))
    plt.plot(history['best_objectives'], label='Best Objective', linewidth=2)
    plt.plot(history['current_objectives'], label='Current Objective', alpha=0.7)
    plt.xlabel('Iteration')
    plt.ylabel('Objective Value')
    plt.title('ALNS Optimization Progress')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()
    
    print("Visualization complete!")
    
except ImportError:
    print("Matplotlib not installed. Skipping visualization.")
    print("Install with: pip install matplotlib")

## Conclusion

In this tutorial, we've demonstrated the complete workflow of the VRP Toolkit:

1. **Setup**: Imported VRP Toolkit modules
2. **Map Creation**: Generated synthetic map with restaurants and customers
3. **Demand Generation**: Created delivery demands across time intervals
4. **PDPTW Instance**: Built complete problem instance with time windows
5. **ALNS Solution**: Solved using Adaptive Large Neighborhood Search
6. **Visualization**: Plotted optimization progress

### Next Steps

- Try with different problem parameters (more vehicles, larger instances)
- Experiment with real-world map data using `RealDataMap`
- Explore other algorithms in the toolkit
- Check out the sensitivity analysis tutorial for parameter tuning

For more information, see the full documentation and other tutorials.